In [44]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))


In [45]:
import glob
import pandas as pd
from scipy.stats import t
from uvv.uvv_collection import UVVCollection
from uvv.uvv_batchcollection import UVVBatchCollection


In [46]:
input_folder = 'data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic/'
files = glob.glob(input_folder + '*.csv')
files[0:5]

['../../data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic\\blphi0001.csv',
 '../../data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic\\blphi0005.csv',
 '../../data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic\\blphi0006.csv',
 '../../data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic\\blphi0007.csv',
 '../../data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic\\blphi0008.csv']

In [47]:
package_dir = 'data/raw_mrdga/neuer_trial/evaluation/single_batch/monophasic/'
evalcol = UVVBatchCollection.from_local(package_dir)
evalcol

[UVVBatchEntry('blphi0001'), UVVBatchEntry('blphi0005'), UVVBatchEntry('blphi0006'), UVVBatchEntry('blphi0007'), UVVBatchEntry('blphi0008'), UVVBatchEntry('blphi0009'), UVVBatchEntry('blphi0010'), UVVBatchEntry('blphi0011'), UVVBatchEntry('blphi0012'), UVVBatchEntry('blphi0017'), UVVBatchEntry('blphi0018'), UVVBatchEntry('blphi0019'), UVVBatchEntry('blphi0020'), UVVBatchEntry('blphi0021'), UVVBatchEntry('blphi0022'), UVVBatchEntry('blphi0023'), UVVBatchEntry('blphi0024'), UVVBatchEntry('blphio0001'), UVVBatchEntry('blphio0002'), UVVBatchEntry('blphio0003'), UVVBatchEntry('blphio0004'), UVVBatchEntry('blphio0005'), UVVBatchEntry('blphio0006'), UVVBatchEntry('blphio0007'), UVVBatchEntry('blphio0008'), UVVBatchEntry('ln28_10001'), UVVBatchEntry('ln28_20001'), UVVBatchEntry('ln28_20002'), UVVBatchEntry('ln28o0001'), UVVBatchEntry('ln28o0002'), UVVBatchEntry('ln28o0003'), UVVBatchEntry('ln28o0004'), UVVBatchEntry('ln28sba_10001'), UVVBatchEntry('ln28sba_10002'), UVVBatchEntry('ln28sba_10003

In [48]:
package_dir = 'data/raw_mrdga/neuer_trial/exp/'
uvcoll = UVVCollection.from_local(package_dir).remove_repeated()
uvcoll_repeated = UVVCollection.from_local(package_dir) # If these two are not the same, there are repeated files in the collection (which should already have been removed)
print(len(uvcoll))
print(len(uvcoll_repeated))


1124
1141


# Defining the materials, wavelengths, loadings, sonications, and dates you want to use as filter criteria

In [49]:
filter_criteria = uvcoll.get_filter_criteria()
filter_criteria

{'materials': ['BLPHI4',
  'LN31',
  'LN28_1',
  'LN28',
  'LN55',
  'LN28_2',
  'LN46',
  'LN31_1',
  'LN33',
  'LN28SBA_1',
  'LN31_2',
  'BLPHI'],
 'wavelengths': [450, 365, 406],
 'loadings': [0.75, 1.0, 2.0, 1.5, 0.25, '73.5h', 0.5],
 'sonications': [None, 10],
 'phases': ['aqueous']}

In [50]:
for entry in evalcol:
    print(entry.df)
    if 'c(H2O2)' in entry.df.columns: #and hasattr(entry, 'baseline'): # I do not get why this is necessary, but it works
        if entry.df['c(H2O2)'].notnull().all():
            entry.df['c(H2O2)_baseline'] = entry.df['c(H2O2)'] - entry.baseline
        else:
            print(f"Entry {entry.identifier} has missing values in 'c(H2O2)' column.")
    else:
        print(f"Entry {entry.identifier} does not have the required columns or attributes.")

combined_df = pd.concat([entry.df for entry in evalcol if 'c(H2O2)_baseline' in entry.df.columns], ignore_index=True)
print(combined_df)


                        identifier  time    abs420  dilutionfactor  \
0    20240229_uvv_alla_blphi0001_1   0.0  0.089272             2.0   
1    20240229_uvv_alla_blphi0001_2   1.0  0.389035             2.0   
2    20240229_uvv_alla_blphi0001_3   2.0  0.477749             2.0   
3    20240229_uvv_alla_blphi0001_4   3.0  0.446437             2.0   
4    20240229_uvv_alla_blphi0001_5   4.0  0.433074             2.0   
5    20240229_uvv_alla_blphi0001_6   6.0  0.413520             2.0   
6    20240229_uvv_alla_blphi0001_7  24.0  0.408581             2.0   
7    20240229_uvv_alla_blphi0002_1   0.0  0.014858             2.0   
8    20240229_uvv_alla_blphi0002_2   1.0  0.329365             2.0   
9    20240229_uvv_alla_blphi0002_3   2.0  0.394820             2.0   
10   20240229_uvv_alla_blphi0002_4   3.0  0.409775             2.0   
11   20240229_uvv_alla_blphi0002_5   4.0  0.400719             2.0   
12   20240229_uvv_alla_blphi0002_6   6.0  0.433698             2.0   
13   20240229_uvv_al

In [51]:
from scipy.stats import t, norm

# Group the DataFrame by the desired columns
grouped = combined_df.groupby(['material_name', 'exc_wavelength', 'loading', 'sonication', 'phase', 'synthesis_date_of_material'])
if 'sonication' in combined_df.columns:
    #print("There is a sonication column.")
    combined_df['sonication'] = combined_df['sonication'].fillna(0).astype(int)
    #if 10 in combined_df['sonication']:
        #print(combined_df['sonication'])

# Create a dictionary to store the separated DataFrames
separated_dfs = {}
# Iterate through the groups and store each group as a separate DataFrame
for (material, wavelength, loading, sonication, phase, date), group in grouped:
    key = f"{material}_{wavelength}nm_{str(loading).replace('.', '-')}_g_L_{str(sonication).replace('.','-')}min_{(phase)}_{date}"
    separated_dfs[key] = group.reset_index(drop=True)
for key, df in separated_dfs.items():
    wavelength = int(df['exc_wavelength'].iloc[0])  # Extract wavelength
    loading = df['loading'].iloc[0]  # Extract loading
    
for key, df in separated_dfs.items():
    times = pd.unique(df['time'])
    meanlist = []
    stdlist = []
    std_err_list = []
    measurements = []
    for time in times:
        def grubbs_test(data, alpha=0.05):
            n = len(data)
            mean = data.mean()
            std_dev = data.std()
            if std_dev != 0:
                G = max(abs(data - mean)) / std_dev
            else:
                G = 0  # Consider the value valid if std_dev is 0
            t_critical = t.ppf(1 - alpha / (2 * n), n - 2)
            G_critical = ((n - 1) / (n ** 0.5)) * ((t_critical ** 2) / (n - 2 + t_critical ** 2)) ** 0.5
            return G, G_critical

        values = df[df.time == time]['c(H2O2)_baseline']
        outliers = []
        while True:
            G, G_critical = grubbs_test(values)
            if G > G_critical:
                outlier = values[abs(values - values.mean()).idxmax()]
                outliers.append(outlier)
                values = values[values != outlier]
            else:
                break
        df.loc[df.time == time, 'Grubbs test'] = df.loc[df.time == time, 'c(H2O2)_baseline'].apply(lambda x: 'FALSE' if x in outliers else 'TRUE')
        
        meanlist.append(values.mean())
        n = len(values)
        measurements.append(n)
        if n > 1:
            std_err = values.std() / (n ** 0.5)
            std_err_list.append(std_err)
            confidence_interval = t.ppf(1 - 0.05 / 2, n - 1) * std_err  # 95% confidence interval
            stdlist.append(confidence_interval)
        else:
            stdlist.append(float('nan'))
            std_err_list.append(float('nan'))

    print(f"Mean list for {key}: {meanlist}")
    print(f"Standard deviation list for {key}: {stdlist}")
    print(f"DataFrame for {key}:\n{df}")
    data = {'times': times, 'average_c': meanlist, '95% confidence interval': stdlist, 'standard error': std_err_list, 'number of datapoints': measurements}
    result_df = pd.DataFrame(data)
    result_df
    wavelength = int(df['exc_wavelength'].iloc[0])  # Extract wavelength
    loading = df['loading'].iloc[0]  # Extract loading
    df.to_csv(f"data/raw_mrdga/neuer_trial/evaluation/material_excwavelength_loading_sonication/{key}_calculatedfrom.csv", index=False)
    result_df.to_csv(f"data/raw_mrdga/neuer_trial/evaluation/material_excwavelength_loading_sonication/{key}.csv", index=False)


Mean list for LN28SBA_1_450nm_0-75_g_L_10-0min_aqueous_202401xx: [0.0]
Standard deviation list for LN28SBA_1_450nm_0-75_g_L_10-0min_aqueous_202401xx: [nan]
DataFrame for LN28SBA_1_450nm_0-75_g_L_10-0min_aqueous_202401xx:
                          identifier time    abs420  dilutionfactor  \
0  20240418_uvv_alla_ln28sba_10003_1  0.0  0.003615             2.0   

   exc_wavelength loading material_name synthesis_date_of_material  \
0             450    0.75     LN28SBA_1                   202401xx   

   sonication  purging purging gas    phase   c(H2O2)  c(H2O2)_baseline  \
0          10     10.0          O2  aqueous  0.011851               0.0   

  Grubbs test  
0        TRUE  
Mean list for LN31_1_450nm_0-75_g_L_10-0min_aqueous_202401xx: [-0.0090681237819672, 1.1283245781074647, 2.6972973562710383, 4.3018084264284155, 5.992081401510383, 8.429670882548633, 30.05956286415082]
Standard deviation list for LN31_1_450nm_0-75_g_L_10-0min_aqueous_202401xx: [0.11522143734898417, 4.11699485604

## Make an evaluation ignoring the synthesis date

In [52]:

# Normalizing material names first, then grouping
# Normalizing material names, treating any name starting with "BLPHI" as "BLPHI4" 
def norm_material(name):
    return 'BLPHI4' if str(name).upper().startswith('BLPHI') else name

grouped = (
    combined_df
    .assign(material_name=combined_df['material_name'].apply(norm_material))
    .groupby(['material_name', 'exc_wavelength', 'loading', 'sonication', 'phase'])
)

if 'sonication' in combined_df.columns:
    #print("There is a sonication column.")
    combined_df['sonication'] = combined_df['sonication'].fillna(0).astype(int)
    #if 10 in combined_df['sonication']:
        #print(combined_df['sonication'])
      
# Create a dictionary to store the separated DataFrames
separated_dfs = {}
# Iterate through the groups and store each group as a separate DataFrame
for (material, wavelength, loading, sonication, phase), group in grouped:
    normed_material = norm_material(material)
    key = f"{normed_material}_{wavelength}nm_{str(loading).replace('.', '-')}_g_L_{str(sonication).replace('.','-')}min_{(phase)}"
    separated_dfs[key] = group.reset_index(drop=True)
for key, df in separated_dfs.items():
    wavelength = int(df['exc_wavelength'].iloc[0])  # Extract wavelength
    loading = df['loading'].iloc[0]  # Extract loading
    
for key, df in separated_dfs.items():
    times = pd.unique(df['time'])
    meanlist = []
    stdlist = []
    std_err_list = []
    measurements = []
    for time in times:
        def grubbs_test(data, alpha=0.05):
            n = len(data)
            mean = data.mean()
            std_dev = data.std()
            if std_dev != 0:
                G = max(abs(data - mean)) / std_dev
            else:
                G = 0  # Consider the value valid if std_dev is 0
            t_critical = t.ppf(1 - alpha / (2 * n), n - 2)
            G_critical = ((n - 1) / (n ** 0.5)) * ((t_critical ** 2) / (n - 2 + t_critical ** 2)) ** 0.5
            return G, G_critical

        values = df[df.time == time]['c(H2O2)_baseline']
        outliers = []
        while True:
            G, G_critical = grubbs_test(values)
            if G > G_critical:
                outlier = values[abs(values - values.mean()).idxmax()]
                outliers.append(outlier)
                values = values[values != outlier]
            else:
                break
        df.loc[df.time == time, 'Grubbs test'] = df.loc[df.time == time, 'c(H2O2)_baseline'].apply(lambda x: 'FALSE' if x in outliers else 'TRUE')
        
        meanlist.append(values.mean())
        n = len(values)
        measurements.append(n)
        if n > 1:
            std_err = values.std() / (n ** 0.5)
            std_err_list.append(std_err)
            confidence_interval = t.ppf(1 - 0.05 / 2, n - 1) * std_err  # 95% confidence interval
            stdlist.append(confidence_interval)
        else:
            stdlist.append(float('nan'))
            std_err_list.append(float('nan'))

    print(f"Mean list for {key}: {meanlist}")
    print(f"Standard deviation list for {key}: {stdlist}")
    print(f"DataFrame for {key}:\n{df}")
    data = {'times': times, 'average_c': meanlist, '95% confidence interval': stdlist, 'standard error': std_err_list, 'number of datapoints': measurements}
    result_df = pd.DataFrame(data)
    result_df
    wavelength = int(df['exc_wavelength'].iloc[0])  # Extract wavelength
    loading = df['loading'].iloc[0]  # Extract loading
    df.to_csv(f"data/raw_mrdga/neuer_trial/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/{key}_calculatedfrom.csv", index=False)
    result_df.to_csv(f"data/raw_mrdga/neuer_trial/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/{key}.csv", index=False)


Mean list for BLPHI4_365nm_0-75_g_L_0min_aqueous: [0.0, 3.6712187594163934, 7.358282213135917, 10.494482360455189, 13.135035228433333, 17.210043964881418, 28.135437797820263]
Standard deviation list for BLPHI4_365nm_0-75_g_L_0min_aqueous: [0.0, 0.20089453553391395, 0.3956382758342981, 0.5629088222887487, 0.7668057948542406, 1.3110761614980975, 2.951972919975453]
DataFrame for BLPHI4_365nm_0-75_g_L_0min_aqueous:
                        identifier  time    abs420  dilutionfactor  \
0    20240523_uvv_alla_blphi0021_1   0.0  0.006172             2.0   
1    20240523_uvv_alla_blphi0021_2   1.0  0.390151             6.0   
2    20240523_uvv_alla_blphi0021_3   2.0  0.490470            10.0   
3    20240523_uvv_alla_blphi0021_4   3.0  0.497592            14.0   
4    20240523_uvv_alla_blphi0021_5   4.0  0.386794            18.0   
..                             ...   ...       ...             ...   
80  20240228_uvv_alla_blphio0008_3   2.0  0.455913            10.0   
81  20240228_uvv_alla_blp

## Plan for the biphasic system

### So, in principle here is what I want to do here:
- I want to add the criterium "biphasic" to my yaml file (Note, that only the solvent is biphasic, the system can be triphasic or more complex)
- then I want to save this information to the dataframe I have created.
- Now, I need to get a criterium of that combines the two dfs to one even though the phase itself is not the same (but everything else should be the same, except the abs value I get). Then I calculate the number of moles in both phases, combined and singular and also put this in the dataframe